# Lesson 12 — Human/AI Task Split and Value Protection

## Goal

Design AI workflows that create value without increasing operational risk.
Learn to map AI capabilities to human decision gates, build failure-mode mitigations,
and implement control points that protect both value and accountability.

---

## Learning Objectives

By the end of this lesson, you will:

1. **Classify AI tasks** — Distinguish retrieve, summarize, extract, classify, draft, recommend tasks
2. **Define human gates** — Know when humans must approve external comms, financial transactions, legal judgment, access changes, exceptions
3. **Build decision-rights matrices** — Map workflows to AUTOMATED, SUPERVISED, APPROVED, and HUMAN-DECIDES quadrants
4. **Identify failure modes** — Recognize 7 AI failure modes and design mitigations for each
5. **Implement control points** — Log decisions, gate approvals, build rollback paths
6. **Quantify value with oversight cost** — Show that human approval overhead is justified by confidence gain

---

## Prerequisites

- **Lesson 06:** Cost of Friction (EUR values per workflow)
- **Lesson 07:** Value Stream Metrics (bottleneck identification)
- **Lesson 08:** Graph Bottlenecks to Value (structural role of nodes)
- **Lesson 11:** AI Opportunity Scoring (15 opportunities with scores)

---

## Core Insight

**AI amplifies human decisions; humans remain accountable.**
A well-designed AI workflow removes friction while preserving auditability.

## Part 1 — AI Capabilities and Human Gates

### Eight AI Task Classes

| Task Class | Example | AI Can | Human Must |
|---|---|---|---|
| **Retrieve** | Fetch invoice from document store | Read past data, search | Verify relevance to current case |
| **Summarize** | Condense ticket history to one paragraph | Extract key facts, compress | Confirm summary accuracy before action |
| **Extract** | Pull vendor name, amount, date from invoice | Parse structured fields | Review for anomalies (typos, duplicates) |
| **Classify** | Route support ticket by intent (billing, technical, urgent) | Match patterns, assign scores | Approve classification if score < 0.8 |
| **Draft** | Write response to customer inquiry | Generate natural language | Always review & edit before send (external comms rule) |
| **Compare** | Show variance between PO and invoice | Compute diffs, highlight mismatches | Decide if variance is acceptable |
| **Recommend** | Suggest next action (approve, escalate, hold) | Rank by risk/value | Decide final action (AI input only) |
| **Prepare** | Assemble pre-approval pack (docs, summary, checklist) | Organize, format, cross-reference | Verify completeness before gate |

### Five Human Decision Gates

These decisions MUST remain with humans, regardless of AI capability:

1. **External Communication** — Any message leaving the organization must be reviewed by a human
2. **Financial Transactions** — Payment, discount, write-off, or amount change must be approved by authorized human
3. **Legal Judgment** — Contract interpretation, regulatory compliance, liability assessment must be decided by humans
4. **High-Risk Access Changes** — Grant, revoke, or escalate permissions must be approved by owner or security
5. **Ambiguous Exceptions** — Cases that do not fit standard rules must go to human judgment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("Libraries loaded")
print("Ready to build human/AI task split frameworks")

---

## Part 2 — Decision-Rights Framework

### Four Decision-Right Quadrants

| Decision Right | AI Role | Human Role | Example |
|---|---|---|---|
| **AUTOMATED** | Execute and log | Monitoring only | Auto-approve invoice if all fields match, amount < 5k |
| **SUPERVISED** | Execute, human monitors | Review during/after | AI suggests dunning level; analyst checks for exceptions |
| **APPROVED** | AI prepares, human approves | Review and sign off before execution | LLM drafts response; analyst reads before send |
| **HUMAN_DECIDES** | AI retrieves & summarizes only | Human makes decision using AI input | Manager chooses between renewal options; AI shows cost |

In [ ]:
def classify_decision_right(task_type, frequency_per_year, reversibility, consequence_severity):
    if frequency_per_year > 1000 and consequence_severity <= 2 and reversibility:
        return "AUTOMATED"
    if task_type in ["draft", "prepare"] and consequence_severity >= 2:
        return "APPROVED"
    if frequency_per_year <= 100 and consequence_severity >= 4:
        return "HUMAN_DECIDES"
    return "SUPERVISED"

print("Decision rights classifier loaded")

---

## Part 3 — Failure Modes and Control Points

### Seven AI Failure Modes

1. **Hallucination** — AI generates plausible but false info
2. **Context Drift** — AI uses outdated rules
3. **Data Poisoning** — Corrupted input data
4. **Permission Escalation** — AI escalates task inappropriately
5. **Audit Loss** — Critical decision not logged
6. **Cascading Error** — One AI mistake triggers downstream errors
7. **User Override** — Human bypasses AI gate without justification


---

## Part 4 — Worked Example 1: Invoice Approval Workflow

### Context from L06 / L08 / L11

- **4,200 invoices/year** (350/month)
- **L06 friction cost:** EUR 308 per invoice cycle (6% of total)
- **L08 graph finding:** Clerk queue (24h wait), Manager bottleneck (centrality 0.4)
- **L11 opportunity:** Invoice triage agent (score 18, MEDIUM priority)

In [ ]:
invoice_tasks = pd.DataFrame({
    "task": ["Scan & OCR", "Classify vendor", "Extract amount", "Flag duplicate", "Compare PO vs Inv", "Route to manager", "Route to accounting", "Generate approval pack", "Log decisions", "Notify submitter"],
    "decision_right": ["AUTOMATED", "SUPERVISED", "SUPERVISED", "APPROVED", "APPROVED", "APPROVED", "AUTOMATED", "APPROVED", "AUTOMATED", "AUTOMATED"],
})

print("INVOICE WORKFLOW -- Task-Split Matrix")
print(invoice_tasks.to_string(index=False))

In [ ]:
time_saved_annual = 4200 * (21/60) * 55  # 21 min saved per invoice
approval_overhead = 4200 * (21/60) * 55 * 0.5  # 50% overhead
net_value = time_saved_annual - approval_overhead - 2000

print(f"Annual time saved:      EUR {time_saved_annual:,.0f}")
print(f"Approval overhead:      EUR {approval_overhead:,.0f}")
print(f"Risk mitigation:        EUR 2,000")
print(f"Net annual value:       EUR {net_value:,.0f}")

In [ ]:
print("Invoice Control Points:")
print("  Gate 1: Flag if duplicate found")
print("  Gate 2: Analyst approval if variance > 5%")
print("  Gate 3: Manager approval if amount > EUR 10k")
print()
print("Rollback: If manager rejects, return to analyst within 30 min")

---

## Part 5 — Worked Example 2: Support Triage Workflow

### Context from L06 / L08 / L11

- **18,000 tickets/year** (1,500/month)
- **L06 friction cost:** EUR 9,603 (27% of total)
- **L08 graph finding:** Specialist bottleneck (highest friction edge)
- **L11 opportunity:** LLM intent classifier (score 22, HIGH priority)

In [ ]:
support_tasks = pd.DataFrame({
    "task": ["Receive", "Extract", "Classify", "Search KB", "Suggest", "Route analyst", "Route specialist", "Draft response", "Send", "Close"],
    "decision_right": ["AUTOMATED", "AUTOMATED", "AUTOMATED", "AUTOMATED", "SUPERVISED", "SUPERVISED", "APPROVED", "APPROVED", "SUPERVISED", "SUPERVISED"],
})

print("SUPPORT WORKFLOW -- Task-Split Matrix")
print(support_tasks.to_string(index=False))

In [ ]:
def support_escalation(intent_confidence, complexity, customer_type):
    if intent_confidence < 0.6:
        return "ESCALATE_TO_ANALYST"
    if complexity >= 4 or customer_type == "vip":
        return "ESCALATE_TO_SPECIALIST"
    if complexity <= 2 and intent_confidence >= 0.85:
        return "AUTO_RESPOND"
    return "ROUTE_TO_ANALYST"

test = support_escalation(0.95, 1, "new")
print(f"Test case (high conf, low complexity, new customer): {test}")

In [ ]:
time_saved_support = 18000 * (12/60) * 55
overhead_support = (180*5 + 2700*3) / 60 * 55
net_support = time_saved_support - overhead_support - 3000

print(f"Annual time saved:      EUR {time_saved_support:,.0f}")
print(f"Approval overhead:      EUR {overhead_support:,.0f}")
print(f"Risk mitigation:        EUR 3,000")
print(f"Net annual value:       EUR {net_support:,.0f}")

---

## Part 6 — Worked Example 3: Management Reporting Workflow

### Context from L06 / L08 / L11

- **52 reports/year** (4/month)
- **L06 friction cost:** EUR 4,022 (44% of total — highest %)
- **L08 graph finding:** Executive centrality 1.0 (100% of reports)
- **L11 opportunity:** LLM pre-review (score 15, MEDIUM priority)

In [ ]:
reporting_tasks = pd.DataFrame({
    "task": ["Ingest data", "Validate", "Draft", "Check tone", "Summary", "Prepare", "Track revisions", "Publish"],
    "decision_right": ["AUTOMATED", "APPROVED", "APPROVED", "APPROVED", "APPROVED", "APPROVED", "APPROVED", "HUMAN_DECIDES"],
})

print("REPORTING WORKFLOW -- Task-Split Matrix")
print(reporting_tasks.to_string(index=False))

In [ ]:
print("REPORTING -- Critical Control Points (High-Governance Domain)")
print()
print("✓ Data Ingestion:     Automated if schema valid (logging: source + row count)")
print("✓ Validation Gate:    Human (analyst) if anomalies (logging: all edits)")
print("✓ Draft Review:       Human (analyst) edits (logging: timestamp + author)")
print("✓ Executive Approval: Human (executive) signs (logging: signature)")
print("✓ Distribution:       Human (executive) authorizes (logging: recipients)")

---

## Part 7 — Unified Decision-Rights Framework

### The 3×3×2 Taxonomy

**Decision Frequency:** High (>1000/yr) | Medium (100–1000/yr) | Low (<100/yr)
**Reversibility:** Reversible | Irreversible
**Consequence (1–5):** 1–2 (low) | 3 (medium) | 4–5 (high)

### Typical Portfolio Distribution

- **40% AUTOMATED** — High-frequency, low-consequence, reversible
- **40% APPROVED** — Medium-frequency, medium-consequence, mostly reversible
- **20% HUMAN_DECIDES** — Low-frequency, high-consequence, many irreversible

In [ ]:
all_30_tasks = pd.concat([
    invoice_tasks.assign(workflow="Invoice"),
    support_tasks.assign(workflow="Support"),
    reporting_tasks.assign(workflow="Reporting"),
], ignore_index=True)

print("AGGREGATE -- Distribution by Decision Right")
print(all_30_tasks.groupby("decision_right")["task"].count())
print()
dist = all_30_tasks["decision_right"].value_counts()
for dr, count in dist.items():
    pct = 100 * count / len(all_30_tasks)
    print(f"{dr:20s}: {count:2d} tasks ({pct:4.1f}%)")

In [ ]:
invoice_net = time_saved_annual - approval_overhead - 2000
support_net = net_support
reporting_net = 52 * (40/60) * 75 - 2000 - 1500

total_portfolio = invoice_net + support_net + reporting_net

print("CROSS-WORKFLOW COST-BENEFIT")
print("="*70)
print(f"Invoice Net Value:      EUR {invoice_net:,.0f}/year")
print(f"Support Net Value:      EUR {support_net:,.0f}/year")
print(f"Reporting Net Value:    EUR {reporting_net:,.0f}/year")
print()
print(f"TOTAL PORTFOLIO:        EUR {total_portfolio:,.0f}/year")

---

## Part 8 — Exercises

### Exercise 1: Guided — PO Approval Workflow

Classify 8 procurement tasks to decision rights:
1. Receive PO from requestor
2. Validate vendor in masterfile
3. Check budget availability
4. Compare to historical pricing
5. Flag if price variance > 10%
6. Route to approver (amount-based)
7. Generate purchase order
8. Send to vendor & confirm

In [ ]:
ex1_answer = {
    "tasks": ["Receive", "Validate vendor", "Check budget", "Compare price", "Flag variance", "Route approver", "Generate PO", "Send vendor"],
    "decision_right": ["AUTOMATED", "SUPERVISED", "APPROVED", "SUPERVISED", "APPROVED", "APPROVED", "AUTOMATED", "APPROVED"],
}

print("EXERCISE 1 -- PO Workflow Task-Split")
for t, dr in zip(ex1_answer["tasks"], ex1_answer["decision_right"]):
    print(f"  {t:25s} → {dr}")

print()
print("Key insight: Financial transaction (PO) requires approval gates at:")
print("  - Budget check (ensure fund availability)")
print("  - Price variance (flag unusual spend)")
print("  - Approver routing (amount-based authorization)")

---

### Exercise 2: Open-Ended — Collections Prioritization

Design a human/AI task split for DSO (Days Sales Outstanding) reduction:
1. Identify overdue invoices
2. Score payment risk
3. Score customer importance
4. Calculate urgency
5. Suggest collection action
6. Apply discount (if approved)
7. Send collection message
8. Log to working-capital P&L

Deliverable:
- Classify 8–10 tasks to decision rights
- Identify 2–3 failure modes with mitigations
- Design 2 control points
- Estimate: time saved, approval overhead, net value

In [ ]:
ex2_answer = {
    "task": ["Identify overdue", "Score risk", "Score importance", "Calculate urgency", "Suggest action", "Route collector", "Offer discount", "Send message"],
    "decision_right": ["AUTOMATED", "SUPERVISED", "SUPERVISED", "AUTOMATED", "APPROVED", "SUPERVISED", "APPROVED", "APPROVED"],
}

print("EXERCISE 2 -- Collections Workflow")
for t, dr in zip(ex2_answer["task"], ex2_answer["decision_right"]):
    print(f"  {t:25s} → {dr}")

print()
print("Key Failure Modes:")
print("  1. Over-escalation (VIP customer flagged as risky)")
print("     Mitigation: Collector reviews VIP status; override gate")
print()
print("  2. Unauthorized discount (AI suggests discount without approval)")
print("     Mitigation: Discount always requires manager approval")
print()
print("Value: EUR 260k time saved - EUR 39k overhead = EUR 221k net")

---

### Exercise 3: Capstone — ML Access Control Anomaly Detection

Design human/AI task split for IAM (Identity & Access Management) risk detection:
1. Monitor user activity real-time
2. Detect anomalies (ML model)
3. Calculate risk severity
4. Route to security team if threshold exceeded
5. Execute remediation (block/revoke)
6. Log all alerts & decisions

Deliverable:
- Design 12+ tasks across detection, triage, escalation, logging
- Classify each to decision right
- Define 4–5 failure modes (high-consequence domain!)
- Design 3–4 control points (approval gates, rollback paths)
- Quantify: capacity freed, audit effort saved, risk reduction
- Write PE memo on governance importance

In [ ]:
print("EXERCISE 3 -- Access Control Anomaly Detection")
print("="*70)
print()
ex3_tasks = {
    "Ingest activity logs": "AUTOMATED",
    "Detect anomalies": "AUTOMATED",
    "Calculate risk (1-5)": "AUTOMATED",
    "Escalate to analyst": "APPROVED",
    "Escalate to security": "APPROVED",
    "Execute remediation": "APPROVED",
    "Audit logging": "AUTOMATED",
}

for task, dr in ex3_tasks.items():
    print(f"  {task:30s} → {dr}")

print()
print("Critical Failure Modes (High-Consequence):")
print("  1. False positive → blocks business operation")
print("     Mitigation: Analyst reviews before revocation; rollback within 1 hour")
print()
print("  2. Insider threat becomes known → attacker covers tracks")
print("     Mitigation: No notification until security officer approves disclosure")
print()
print("  3. Privacy violation in alert → GDPR risk")
print("     Mitigation: Alerts encrypted; compliance review before notification")
print()
print("Value: EUR 693k analyst capacity freed + EUR 19k audit savings = EUR 712k total")
print()
print("Key Governance Principle:")
print("  High-consequence domain requires MORE human oversight, not less.")
print("  Approval gates protect both value AND auditability.")